# PlantCLEF2026 Baseline with MLflow
This notebook reproduces the tiling inference baseline and tracks the experiment using the local MLflow and MinIO setup.

In [ ]:
import sys
import os

# Add the src folder to the Python path
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import timm
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import time
import torchvision.transforms as T
from torch.amp import autocast
from matplotlib import pyplot as plt
from kornia import tensor_to_image
from kornia.contrib import extract_tensor_patches, compute_padding
import csv
import mlflow

from src.config.mlflow_init import init_mlflow

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

# Initialize MLflow experiment using the loaded config
init_mlflow()

In [ ]:
from src.utils.metrics import AverageMeter
from src.data.datasets import PatchDataset, TestDataset

In [ ]:
from src.data.metadata import load_metadata

data_dir = os.environ.get("DATA_DIR", "../data")
df_species_ids, df_metadata, class_map = load_metadata(data_dir)

df_metadata.head()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "vit_base_patch14_reg4_dinov2.lvd142m"
model_dir = os.environ.get("MODEL_DIR", "../model")

# Initialize model
model = timm.create_model(
    model_name,
    pretrained=False,
    num_classes=len(df_species_ids),
    checkpoint_path=f"{model_dir}/model_best.pth.tar",
)  # Directed to the extracted model folder
model = model.to(device)
model = model.eval()

data_config = timm.data.resolve_model_data_config(model)
model_input_size, model_mean, model_std = (
    data_config["input_size"][1],
    data_config["mean"],
    data_config["std"],
)

In [ ]:
# Hyperparameters
batch_size = 64
min_score = 0.1
top_k_tile = 5
patch_size = model_input_size
stride = int(model_input_size / 2)
use_pad = True

Submit prediction inside an MLflow run to log metrics and artifacts

In [ ]:
# Setup PyTorch Dataset and DataLoader
image_to_tensor = T.ToTensor()
data_dir = os.environ.get("DATA_DIR", "../data")
dataset = TestDataset(
    image_folder=f"{data_dir}/PlantCLEF2025_test_images/PlantCLEF2025_test_images/",
    patch_size=patch_size,
    stride=stride,
    use_pad=True,
    transform=image_to_tensor,
)
dataloader = DataLoader(dataset, batch_size=1, num_workers=4, pin_memory=True)

image_predictions = {}
image_probabilities = (
    {}
)  # Record raw probabilities for potential advanced metrics later
batch_time = AverageMeter()

# Enable PyTorch autologging and MLflow tracing if available
mlflow.pytorch.autolog()

# Start MLflow run
with mlflow.start_run(run_name="tiling-inference-baseline") as run:
    # Log parameters
    mlflow.log_params(
        {
            "model_name": model_name,
            "batch_size": batch_size,
            "min_score": min_score,
            "top_k_tile": top_k_tile,
            "patch_size": patch_size,
            "stride": stride,
            "use_pad": use_pad,
            "device": device.type,
            "dataset_size": len(dataset),
        }
    )

    # Log the full loaded model into MLflow artifact store
    mlflow.pytorch.log_model(model, "vit_dino_model")

    end = time.time()

    with torch.no_grad():
        for batch_idx, (patches, image_path) in enumerate(dataloader):
            image_results = {}
            quadrat_id = os.path.splitext(os.path.basename(image_path[0]))[0]
            transform_patch = T.Normalize(mean=model_mean, std=model_std)
            patch_dataset = PatchDataset(patches[0], transform=transform_patch)
            patch_loader = DataLoader(
                patch_dataset, batch_size=batch_size, shuffle=False
            )

            for batch_patches in patch_loader:
                batch_patches = batch_patches.to(device)

                with autocast(device.type):
                    outputs = model(batch_patches)
                    probabilities = torch.nn.functional.softmax(outputs, dim=1)

                    top_probs, top_indices = torch.topk(probabilities, top_k_tile)
                    top_probs = top_probs.cpu().numpy()
                    top_indices = top_indices.cpu().numpy()

                    for top_tile_indices, top_tile_probs in zip(top_indices, top_probs):
                        for top_idx, top_prob in zip(top_tile_indices, top_tile_probs):
                            species_id = class_map[top_idx]
                            if top_prob > min_score:
                                if (
                                    species_id not in image_results
                                    or image_results[species_id] < top_prob
                                ):
                                    image_results[species_id] = top_prob

            # Predict list
            predictions_list = list(image_results.keys())
            image_predictions[quadrat_id] = predictions_list
            image_probabilities[quadrat_id] = image_results

            # Batch Tracking
            num_predictions = len(predictions_list)
            max_prob = max(image_results.values()) if image_results else 0.0

            batch_time.update(time.time() - end)
            end = time.time()

            # Step metrics
            mlflow.log_metric("step_batch_time", batch_time.val, step=batch_idx)
            mlflow.log_metric(
                "num_predictions_per_image", num_predictions, step=batch_idx
            )
            mlflow.log_metric("max_prob_per_image", max_prob, step=batch_idx)

            if batch_idx % 10 == 0:
                print(
                    f"Predict: [{batch_idx}/{len(dataloader)}] Time {batch_time.val:.3f} ({batch_time.avg:.3f})"
                )

    # Log global metrics
    mlflow.log_metric("avg_batch_time_seconds", batch_time.avg)
    mlflow.log_metric("total_images_processed", len(dataloader))

    # Save predictions
    submission_path = "submission.csv"
    df_run = pd.DataFrame(
        list(image_predictions.items()), columns=["quadrat_id", "species_ids"]
    )
    df_run["species_ids"] = df_run["species_ids"].apply(str)
    df_run.to_csv(submission_path, sep=",", index=False, quoting=csv.QUOTE_ALL)

    # Store submission.csv artifact
    mlflow.log_artifact(submission_path)

    # ==========================================
    # EVALUATION METRICS & PLOTS (Optional)
    # ==========================================
    # If evaluating on a validation set with known ground truth, compute the
    # PlantCLEF official metric (Macro-averaged F1 per sample) along with others.
    import ast
    import seaborn as sns
    from sklearn.preprocessing import MultiLabelBinarizer
    from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

    ground_truth_path = f"{data_dir}/validation_ground_truth.csv"  # Switch to your validation set's truth path

    if os.path.exists(ground_truth_path):
        print("Ground truth found. Computing metrics...")
        df_gt = pd.read_csv(ground_truth_path)

        # Parse arrays properly from strings and ensure they are always lists
        def _parse_species(x):
            if isinstance(x, str):
                x = ast.literal_eval(x)
            return x if isinstance(x, list) else [x]

        df_gt["species_ids"] = df_gt["species_ids"].apply(_parse_species)

        # Build predicted species lists directly from in-memory dict
        # (avoids fragile str/literal_eval roundtrip that breaks on NumPy 2.0+)
        df_run["predicted_species_ids"] = df_run["quadrat_id"].map(image_predictions)

        # Merge predictions with Ground Truth
        df_eval = pd.merge(
            df_gt,
            df_run[["quadrat_id", "predicted_species_ids"]],
            on="quadrat_id",
            how="inner",
        )

        print(
            "Evaluation join summary -> "
            f"ground truth rows: {len(df_gt)} | predictions rows: {len(df_run)} | matched rows: {len(df_eval)}"
        )

        if df_eval.empty:
            print(
                "Skipping metric computation because no quadrat_id matched between "
                "validation_ground_truth.csv and model predictions."
            )
        else:
            # Convert explicit list format into MLB binary dense format to compute Scikit-Learn metrics
            mlb = MultiLabelBinarizer()
            all_labels = pd.concat(
                [df_eval["species_ids"], df_eval["predicted_species_ids"]]
            )
            mlb.fit(all_labels)

            y_true = mlb.transform(df_eval["species_ids"])
            y_pred = mlb.transform(df_eval["predicted_species_ids"])

            # Calculate Macro-averaged metrics explicitly per sample
            f1_samples = f1_score(y_true, y_pred, average="samples", zero_division=0)
            precision_samples = precision_score(
                y_true, y_pred, average="samples", zero_division=0
            )
            recall_samples = recall_score(
                y_true, y_pred, average="samples", zero_division=0
            )

            # ROC AUC proxy computation based on output thresholds
            try:
                roc_auc = roc_auc_score(y_true, y_pred, average="macro")
            except ValueError:
                roc_auc = float(
                    "nan"
                )  # Failsafe if class variations are insufficient in batch

            # Log aggregated validation scalars to MLflow
            mlflow.log_metrics(
                {
                    "val_f1_score_samples": f1_samples,
                    "val_precision_samples": precision_samples,
                    "val_recall_samples": recall_samples,
                    "val_roc_auc_macro": roc_auc,
                }
            )

            # Visualize the metrics in a bar plot
            fig, ax = plt.subplots(figsize=(8, 6))
            metrics_df = pd.DataFrame(
                {
                    "Metric": [
                        "F1 (Samples)",
                        "Precision (Samples)",
                        "Recall (Samples)",
                        "ROC AUC (Macro)",
                    ],
                    "Score": [f1_samples, precision_samples, recall_samples, roc_auc],
                }
            )

            sns.barplot(
                x="Metric", y="Score", data=metrics_df, ax=ax, palette="viridis"
            )
            ax.set_ylim(0, 1.0)
            ax.set_title("Validation Metrics against Ground Truth")

            # Append exact values on top of bars
            for p in ax.patches:
                ax.annotate(
                    f"{p.get_height():.3f}",
                    (p.get_x() + p.get_width() / 2.0, p.get_height()),
                    ha="center",
                    va="bottom",
                )

            # Save figure internally & pass safely into MLflow Artifact storage
            plot_path = "validation_metrics.png"
            fig.savefig(plot_path)
            mlflow.log_artifact(plot_path)
            plt.close(fig)

            print(
                f"Metrics Logged -> F1: {f1_samples:.4f} | Precision: {precision_samples:.4f} | Recall: {recall_samples:.4f}"
            )
    else:
        print(
            f"Skipping evaluation metrics block. Ground truth not found at '{ground_truth_path}'."
        )

    print(f"Run completed. Check MLflow UI at {os.environ['MLFLOW_TRACKING_URI']}")